Writing producer.py


In [3]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
   
    is_fraud = random.random() < 0.05
    
    if is_fraud:
        tx_id = f'FRAUD{random.randint(1000,9999)}'
        amount = round(random.uniform(3000.01, 10000.0), 2)
        category = 'elektronika'
        hour = random.randint(0, 5)
    else:
        tx_id = f'TX{random.randint(1000,9999)}'
        amount = round(random.uniform(5.0, 3000.0), 2)
        category = random.choice(kategorie)
        hour = random.randint(6, 23)

    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
        'hour': hour,
        'is_anomaly': is_fraud
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['is_anomaly']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [9]:
%%file consumer_filter.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)


for message in consumer:
    tx = message.value

    if tx['amount'] > 1000:

        producer.send('filtered_transactions', value=tx)
        print(f" Alert!")
        print(f"ID: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['is_anomaly']}")
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT



Overwriting consumer_filter.py


In [13]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:

        tx['risk_level'] = "High"
    elif tx['amount'] > 1000:

        tx['risk_level'] = "Medium"
    else:
        tx['risk_level'] = "Low"
        
    print(f" {tx['risk_level']} ID:{tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']} | {tx['is_anomaly']}")
    

Overwriting consumer_enrich.py


In [15]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. store_counts[store] += 1
#   2. total_amount[store] += amount
#   3. Co 10 wiadomości: print tabela
for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    
    store_counts[store] += 1
    total_amount[store] += amount
    msg_count += 1
    
    if msg_count % 10 == 0:
        print(f" Wynik (po {msg_count} wiadomościach)")
        print(f"{'SKLEP':<12} | {'LICZBA TX':<10} | {'SUMA PLN':<12}")
        print("-" * 40)
        
        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            print(f"{s:<12} | {count:<10} | {total:>10.2f} PLN")
        
        print("-" * 40)

Overwriting consumer_count.py


In [1]:
def score_transaction(tx):
    score = 0
    rules = []
    
    if tx['amount'] > 3000:
        score += 3
        rules.append('R1')
        
    if tx['amount'] > 1500 and tx['category'] == 'Elektronika':
        score += 2
        rules.append('R2')
        
    if tx['hour'] < 6:
        score += 2
        rules.append('R3')
    
    return score, rules

# Test
test_tx = {'tx_id': 'TX999', 'amount': 4500.0, 'category': 'elektronika',
           'timestamp': '2026-04-01T03:15:00','hour': 3}
print(score_transaction(test_tx))  # powinno dać score >= 5

(5, ['R1', 'R3'])


In [8]:
%%file scoring_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json

consumer = KafkaConsumer('filtered_transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='scoring-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

# TWÓJ KOD
# Dla każdej transakcji: scoruj, jeśli >= 3: wyślij do 'alerts' i wypisz ALERT

def score_transaction(tx):
    score = 0
    rules = []

    amount = tx.get('amount', 0)
    category = str(tx.get('category', '')).lower()
    hour = tx.get('hour', 12)

    
    if amount > 3000:
        score += 3
        rules.append('R1')
        
    if amount > 1500 and category == 'elektronika':
        score += 2
        rules.append('R2')
        
    if hour < 6:
        score += 2
        rules.append('R3')
    
    return score, rules

for message in consumer:
    tx = message.value
    
    total_score, broken_rules = score_transaction(tx)
    tx['total_score'] = total_score
    tx['rules'] = broken_rules

    if total_score >= 3:
        print(f" ALERT ID: {tx['tx_id']} | Punkty: {total_score} | Reguły: {broken_rules}")
        
        alert_producer.send('alerts', value=tx)
        

Overwriting scoring_consumer.py
